In [1]:
import kagglehub

In [2]:
import torch
import torch.nn as nn
from torchvision import models,datasets,transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [3]:
path = kagglehub.dataset_download("tongpython/cat-and-dog")

100%|██████████| 218M/218M [00:01<00:00, 222MB/s]

Extracting files...


In [4]:
print(path)

/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1


In [7]:
import os
os.listdir("/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1")


['test_set', 'training_set']

In [8]:
os.listdir("/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/training_set")


['training_set']

In [ ]:
import os
os.listdir("/kaggle/input/cat-and-dog")


['test_set', 'training_set']

In [ ]:
os.listdir("/kaggle/input/cat-and-dog/training_set")


['training_set']

In [9]:
os.listdir("/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/training_set/training_set")


['dogs', 'cats']

In [10]:
TRAIN_DIR = "/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/training_set/training_set"
TEST_DIR  = "/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1/test_set/test_set"


In [19]:
if torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
model=models.resnet18(pretrained=True)
criterion=nn.CrossEntropyLoss()
model.fc=nn.Linear(model.fc.in_features,2)
model=model.to(device)
for param in model.parameters():
    param.requires_grad=False
for param in model.layer4.parameters():
    param.requires_grad=True
for param in model.layer3.parameters():
    param.requires_grad=True
for param in model.fc.parameters():
    param.requires_grad=True
optimize=optim.Adam([{"params":model.layer3.parameters(),"lr":1e-4},{'params':model.layer4.parameters(),'lr':1e-4},{'params':model.fc.parameters(),'lr':1e-3}])
scheduler = optim . lr_scheduler . StepLR ( optimize, step_size =10 , gamma
=0.1)
def dataset_loading():
    train_transform=transforms.Compose([transforms.RandomResizedCrop(224),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    test_transform=transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    train_set=datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    test_set=datasets.ImageFolder(TEST_DIR, transform=test_transform)
    train_loader=DataLoader(train_set,shuffle=True,batch_size=32)
    test_loader=DataLoader(test_set,shuffle=False,batch_size=32)
    return train_loader,test_loader
def train_data(train_loader,model):

    model.train()
    for epoch in range(10):
        loss_per_epoch=0
        total=0
        for images,labels in train_loader:
            images=images.to(device)
            labels=labels.to(device)
            output=model(images)
            loss=criterion(output,labels)
            optimize.zero_grad()
            loss.backward()
            optimize.step()
            loss_per_epoch+=loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1},Loss:{loss_per_epoch/len(train_loader)}")
def test_data(test_loader,model):
  model.eval()
  correct=0
  test_total=0

  with torch.no_grad():
        for images,labels in test_loader:
            images=images.to(device)
            labels=labels.to(device)
            test_total+=labels.size(0)
            output=model(images)
            _,predicted=torch.max(output,1)
            correct+=(predicted==labels).sum().item()



        accuracy=correct/test_total
        print(f"Test Accuracy with test_set:{accuracy*100}%")
def main():
    train_loader,test_loader=dataset_loading()
    train_data(train_loader,model)
    test_data(test_loader,model)
main()




Epoch 1,Loss:0.16778427088610917
Epoch 2,Loss:0.12308872823577478
Epoch 3,Loss:0.09950886053808657
Epoch 4,Loss:0.09595199650621035
Epoch 5,Loss:0.09119829150733995
Epoch 6,Loss:0.09098019580583114
Epoch 7,Loss:0.09372373667371225
Epoch 8,Loss:0.10189430175007755
Epoch 9,Loss:0.09610661127014583
Epoch 10,Loss:0.0791328522317349
Test Accuracy with test_set:98.41819080573406%
